In [1]:
import pandas as pd

df = pd.read_csv("../data/IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
df.shape

(50000, 2)

In [3]:
df["sentiment"] = df["sentiment"].map({
    "positive":1,
    "negative":0
})

In [4]:
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Amit
[nltk_data]     Gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):

    # lowercase
    text = text.lower()

    # remove html
    text = re.sub(r'<.*?>', '', text)

    # remove punctuation
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # remove stopwords
    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [6]:
df["clean_review"] = df["review"].apply(clean_text)

In [7]:
from sklearn.model_selection import train_test_split

X = df["clean_review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [9]:
%pip install tensorflow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(X_train)

In [11]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [13]:
max_len = 200

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len
)

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [15]:
model = Sequential([
    
    Embedding(input_dim=max_words,
              output_dim=128,
              input_length=max_len),

    SimpleRNN(128),

    Dense(1, activation='sigmoid')
])

model.summary()

c:\Users\Amit Gupta\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [17]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [18]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 31s 59ms/step - accuracy: 0.6976 - loss: 0.5590 - val_accuracy: 0.6867 - val_loss: 0.5911
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.8142 - loss: 0.4175 - val_accuracy: 0.8146 - val_loss: 0.4174
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 66ms/step - accuracy: 0.8147 - loss: 0.4219 - val_accuracy: 0.8183 - val_loss: 0.4251
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 24s 48ms/step - accuracy: 0.8823 - loss: 0.2953 - val_accuracy: 0.8281 - val_loss: 0.3977
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 51ms/step - accuracy: 0.9165 - loss: 0.2287 - val_accuracy: 0.8357 - val_loss: 0.4096
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 24s 47ms/step - accuracy: 0.9320 - loss: 0.1878 - val_accuracy: 0.8111 - val_loss: 0.6229
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 24s 47ms/step - accuracy: 0.9042 - loss: 0.2329 - val_accuracy: 0.6870 - val_loss: 0.5844


In [19]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8316 - loss: 0.3981
Accuracy: 0.83160001039505


In [20]:
model.save("../model/imdb_rnn_model.h5")

In [21]:
import pickle

with open("../model/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)